# collection.ipynb

Pull B-class / C-class English Wikipedia articles for DeBERTa-WP-BCLASS.

## Setup: get title lists from Quarry

1. Go to https://quarry.wmcloud.org and log in.
2. New Query, database `enwiki_p`, and run the query in `quarry_b.sql`.
3. Download the result as CSV, save as `b_class_titles.csv` next to this notebook.
4. Run the query in `quarry_c.sql` and save as `c_class_titles.csv`.

## Labeling rules

- **C-class**: requires a fully parsed b1-b6 checklist on the talk page. Rows without a complete checklist are skipped.
- **B-class**: checklist is fetched opportunistically, not required. If no checklist is found (or it's incomplete), the row is kept with all six criteria assumed `yes`. If a *complete* checklist IS found and at least one of the six is not `yes`, the row is **skipped**.

In [8]:
import csv
import json
import os
import re
import time
import requests
import random
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
# Config
N_PER_CLASS = 1000
OUT_PATH = "bclass_data.jsonl"
DELAY = 1.0  # seconds between requests

WIKIMEDIA_TOKEN = os.environ.get("WIKIMEDIA_TOKEN")  # optional, enables LiftWing scoring

API = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "bclass-research-bot/0.1 (research use; contact: jake) requests"
}
LIFTWING_URL = "https://api.wikimedia.org/service/lw/inference/v1/models/enwiki-articlequality:predict"

B_PARAM_RE = re.compile(r"\|\s*b([1-6])\s*=\s*([a-zA-Z]*)", re.IGNORECASE)
B_KEYS = ["b1", "b2", "b3", "b4", "b5", "b6"]

print(f"N_PER_CLASS={N_PER_CLASS}  OUT_PATH={OUT_PATH}  DELAY={DELAY}  LiftWing={'on' if WIKIMEDIA_TOKEN else 'off'}")


N_PER_CLASS=1000  OUT_PATH=bclass_data.jsonl  DELAY=1.0  LiftWing=on


In [10]:
def api_get(params, max_attempts=6):
    """GET against the Action API. On 429, honors the Retry-After header
    if present rather than guessing at a backoff -- a prior run got a
    sustained 429 that 5x exponential backoff never recovered from,
    which means the server was explicitly telling us how long to wait
    and we weren't listening."""
    params = {**params, "format": "json"}
    for attempt in range(max_attempts):
        r = requests.get(API, params=params, headers=HEADERS, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 5 * (attempt + 1)))
            print(f"  [429] rate-limited, waiting {wait}s (attempt {attempt + 1}/{max_attempts})")
            time.sleep(wait)
            continue
        time.sleep(2 * (attempt + 1))
    r.raise_for_status()


def get_wikitext(title):
    data = api_get({
        "action": "query",
        "prop": "revisions",
        "titles": title,
        "rvslots": "main",
        "rvprop": "content|timestamp",
    })
    pages = data.get("query", {}).get("pages", {})
    for p in pages.values():
        revs = p.get("revisions")
        if not revs:
            return None, None
        slot = revs[0]["slots"]["main"]
        return slot.get("*", ""), revs[0].get("timestamp")
    return None, None


def get_plaintext(title):
    data = api_get({
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": title,
    })
    pages = data.get("query", {}).get("pages", {})
    for p in pages.values():
        return p.get("extract", "")
    return ""


def extract_b_flags(wikitext):
    """Return (dict b1..b6 -> normalized value, conflict_bool).
    Returns (None, conflict_bool) if the checklist isn't fully present
    (fewer than 6 well-formed b1..b6 values found)."""
    matches = B_PARAM_RE.findall(wikitext or "")
    if not matches:
        return None, False

    seen = {}
    conflict = False
    for num, val in matches:
        val_norm = val.strip().lower()
        if val_norm in ("y", "yes"):
            val_norm = "yes"
        elif val_norm in ("n", "no"):
            val_norm = "no"
        elif val_norm in ("na", "n/a"):
            val_norm = "na"
        elif val_norm == "":
            continue
        else:
            continue  # junk value like "gobbledygook" -> skip

        key = f"b{num}"
        if key in seen and seen[key] != val_norm:
            conflict = True
        seen[key] = val_norm

    if len(seen) < 6:
        return None, conflict  # incomplete checklist
    return seen, conflict


def get_latest_revid(title):
    data = api_get({"action": "query", "prop": "info", "titles": title})
    pages = data.get("query", {}).get("pages", {})
    for p in pages.values():
        return p.get("lastrevid")
    return None


def query_liftwing(revid):
    """Return dict of class->probability from LiftWing's article quality
    model, or None on failure. Requires WIKIMEDIA_TOKEN to be set."""
    if not WIKIMEDIA_TOKEN or revid is None:
        return None
    try:
        r = requests.post(
            LIFTWING_URL,
            json={"rev_id": revid},
            headers={**HEADERS, "Authorization": f"Bearer {WIKIMEDIA_TOKEN}"},
            timeout=30,
        )
        if r.status_code != 200:
            return None
        data = r.json()
        return data.get("enwiki", {}).get("scores", {}).get(str(revid), {}) \
            .get("articlequality", {}).get("score", {}).get("probability")
    except requests.RequestException:
        return None


def load_quarry_titles(csv_path, limit, seed=None):
    titles = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            t = row.get("page_title") or row.get("title")
            if t:
                titles.append(t.replace("_", " "))
    random.Random(seed).shuffle(titles)
    return titles[:limit]


def build_dataset(cls, article_titles, n, delay=DELAY):
    """
    article_titles: list of ARTICLE titles (namespace 0, from Quarry)
    "Talk:" is prepended per-title below.

    cls == "C": checklist is REQUIRED. Rows without a complete b1..b6
                 block are skipped.
    cls == "B": checklist is fetched OPPORTUNISTICALLY, not required.
                 - No checklist / incomplete checklist -> keep the row,
                   assume all six criteria = \'yes\' (that\'s what class=B
                   means by definition).
                 - Complete checklist found, all six = \'yes\' -> keep,
                   use the real (all-yes) flags.
                 - Complete checklist found, but at least one flag is
                   NOT \'yes\' -> SKIP. Labeled B but the checklist
                   disagrees, so drop it rather than keep a mislabeled row.
    """
    rows = []

    for article_title in article_titles:
        if len(rows) >= n:
            break

        time.sleep(delay)  # throttle every candidate, not just accepted rows

        talk_title = f"Talk:{article_title}"
        wikitext, ts = get_wikitext(talk_title)
        if wikitext is None:
            continue

        flags_full, conflict = extract_b_flags(wikitext)

        if cls == "C":
            if flags_full is None:
                continue
            flags = flags_full
        else:  # cls == "B"
            if flags_full is not None:
                if any(v != "yes" for v in flags_full.values()):
                    continue
                flags = flags_full
            else:
                flags = {k: "yes" for k in B_KEYS}
                ts = None

        text = get_plaintext(article_title)
        if not text or len(text) < 200:
            continue

        revid = get_latest_revid(article_title) if WIKIMEDIA_TOKEN else None
        liftwing_probs = query_liftwing(revid) if revid else None

        rows.append({
            "article_title": article_title,
            "overall_class": cls,
            "assessed_talk_timestamp": ts,
            "b1_referenced": flags["b1"],
            "b2_coverage": flags["b2"],
            "b3_structure": flags["b3"],
            "b4_grammar": flags["b4"],
            "b5_accessible": flags["b5"],
            "b6_supporting_materials": flags["b6"],
            "checklist_present": flags_full is not None,
            "banner_conflict": conflict,
            "revid": revid,
            "liftwing_probs": liftwing_probs,
            "text": text,
        })
        if len(rows) % 50 == 0:
            print(f"  [{cls}] ...{len(rows)} collected")

    return rows


In [11]:
SEED = 230911091605040901 # WIKIPEDIA !!!

b_titles = load_quarry_titles("quarry/b_class_titles.csv", limit=N_PER_CLASS * 3, seed=SEED)
c_titles = load_quarry_titles("quarry/c_class_titles.csv", limit=N_PER_CLASS * 3, seed=SEED)
print(f"loaded {len(b_titles)} B-class candidate titles, {len(c_titles)} C-class candidate titles")

loaded 3000 B-class candidate titles, 3000 C-class candidate titles


In [ ]:
all_rows = []
all_rows += build_dataset("B", b_titles, N_PER_CLASS, DELAY)
all_rows += build_dataset("C", c_titles, N_PER_CLASS, DELAY)

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for row in all_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote {len(all_rows)} rows to {OUT_PATH}")

  [429] rate-limited, waiting 38s (attempt 1/6)
